In [1]:
import contextlib
from datetime import date
from collections import Counter, defaultdict
from collections.abc import Mapping
import re
import os

import periodictable
import pandas as pd
import h5py
import numpy as np

import qcportal
from qcportal.external import scaffold
from qcportal.molecules import Molecule
from qcportal.singlepoint import SinglepointDriver, QCSpecification
from qcportal.optimization import OptimizationSpecification
from qcelemental.models.procedures import OptimizationProtocols
from qcelemental.physical_constants import constants

from enumerate_charge_multiplicity import enumerate_variants

# Fix locale for PostgreSQL
os.environ['LC_ALL'] = 'en_US.UTF-8'
os.environ['LANG'] = 'en_US.UTF-8'

ADDRESS = "https://api.qcarchive.molssi.org:443"
client = qcportal.PortalClient(ADDRESS, cache_dir=".")
from qcfractal.snowflake import FractalSnowflake
# Disable compute workers since we're just creating the dataset, not running calculations
snowflake = FractalSnowflake(compute_workers=0)
client = snowflake.client()

/var/folders/dr/9nnhm1493kv7_s9r_wj_l_jm0000gn/T/ipykernel_17221/195615454.py:18: DeprecationWarning: qcelemental.models.procedures should be accessed through qcelemental.models (or qcelemental.models.v1 or .v2 for fixed QCSchema version). The 'models.procedures' route will be removed as soon as v0.70.0.
  from qcelemental.models.procedures import OptimizationProtocols


In [2]:
#!wget --content-disposition "https://zenodo.org/records/15059433/files/tmqm_xtb_dataset_PdZnFeCu_T100_v1.1.hdf5.gz?download=1"

In [3]:
hdf5 = h5py.File("tmqm_xtb_dataset_PdZnFeCu_T100_v1.1.hdf5", 'r')

## Make QCElemental Molecules

In [4]:
def remove_extraneous_dimension(array):
    shape = list(np.shape(array))
    if 1 in shape:
        shape.remove(1)
    return np.array(array).reshape(shape)

def get_symbols(atomic_numbers):
    return [str(periodictable.elements[x])for x in remove_extraneous_dimension(atomic_numbers)]

def get_molecular_formula(atomic_numbers):
    return "".join([str(y) for x1, x2 in Counter(get_symbols(atomic_numbers)).items() for y in [x1, x2] if y != 1])

def parse_molecular_formula(formula):

    element_counts = defaultdict(int)
    for element, count in re.findall(r'([A-Z][a-z]?)(\d*)', formula):
        element_counts[element] += int(count) if count else 1

    return dict(element_counts)

def get_molecular_weight(atomic_numbers):
    return sum(periodictable.elements[x].mass for x in remove_extraneous_dimension(atomic_numbers))

def merge_dicts(d1, d2):
    return {k: merge_dicts(d1[k], d2[k]) if isinstance(d1.get(k), Mapping) and isinstance(d2.get(k), Mapping) else d2.get(k, d1.get(k)) for k in d1.keys() | d2.keys()}


In [5]:
def apply_mapping(mapping, input_dict, index=0, lx=None):

    output = defaultdict(dict)
    for key, value in mapping.items():
        if isinstance(value, str):
            data = input_dict[value]
            if not isinstance(data, str):
                data = remove_extraneous_dimension(data)
                if lx is not None and len(data) == lx:
                    output[key] = data[index]
                    continue
            output[key] = data
        elif isinstance(value, tuple): # function, input pairs
            output[key] = value[0](*(input_dict[k2] for k2 in value[1:]))
        elif isinstance(value, list):
            output[key].update({k2: input_dict[k2] for k2 in value})
        elif isinstance(value, dict):
            output[key].update(apply_mapping(value, input_dict, index=0, lx=None))
            
    return output
            
def convert_hdf5_group(hdf5_group):
    output = {}
    for key, value in hdf5_group.items():
        if isinstance(value, h5py.Group):
            output[key] = convert_hdf5_group(value)
        elif isinstance(value, h5py.Dataset):
            data = value[()]
            if isinstance(data, np.ndarray):
                output[key] = data
            elif isinstance(data, np.bytes_):
                output[key] = data.decode('utf-8')  # Convert to string
            else:
                output[key] = data.item() if isinstance(data, np.generic) else data  # Convert NumPy scalars
        else:
            output[key] = value

    return output

In [6]:
tm_symbols = ['Pd', 'Fe', 'Zn', 'Cu', 'Mg', 'Li']
hdf5_mapping = {
    "symbols": (get_symbols, "atomic_numbers"), 
    "geometry": "positions",
    "molecular_charge": "total_charge",
    "molecular_multiplicity": "spin_multiplicities",
    "identifiers": {"molecular_formula": (get_molecular_formula, "atomic_numbers"),},
    "extras": {'molecular_weight': (get_molecular_weight, "atomic_numbers")},
}

molecules = defaultdict()
elements, molecular_weights, charges = [], [], []
multiplicities = defaultdict(list)
charges = defaultdict(list)
charge_mult = defaultdict(list)
metals = defaultdict(list)
conformers = Counter()
errors_misc = defaultdict(defaultdict)
errors_mult = defaultdict(list)
for label, mol_hdf5 in hdf5.items():
    mol_dict = convert_hdf5_group(mol_hdf5)
    lx = mol_dict["n_configs"]

    # Get values from HDF5
    qc_input = apply_mapping(hdf5_mapping, mol_dict, index=0, lx=lx) # Only initial value
    qc_input["geometry"] *= 10 / constants.bohr2angstroms # Convert from nm to Bohr (a0)
    qc_input["molecular_multiplicity"] = 1 # All multiplicities are 1 for tmQM
    
    # Get other valid charge/multiplicity variants
    metal = [x for x in qc_input["symbols"] if x in tm_symbols][0]
    variants = enumerate_variants(metal, qc_input["molecular_charge"], qc_input["molecular_multiplicity"])
    charge_mult_pairs = [(v.charge, v.multiplicity) for v in variants]
    charge_mult[f"{label}"] = charge_mult_pairs

    for charge, multiplicity in charge_mult_pairs:
        name = f"{label}-charge={charge}-m{multiplicity}"
        try:
            with open('/dev/null', 'w') as f, contextlib.redirect_stdout(f): # Suppress extraneous print statements
                qc_input["molecular_multiplicity"] = multiplicity
                qc_input["molecular_charge"] = charge
                molecules[name] = Molecule(
                    name=name,
                    fix_com=True,
                    fix_orientation=True,
                    fix_symmetry="c1",
                    comment="Molecule coordinates taken from tmQM.",
                    **qc_input
                )
                multiplicities[f"{label}"].append(multiplicity)
                charges[f"{label}"].append(charge)
            conformers[label] += 1
        except Exception as e:
            if "Inconsistent or unspecified chg/mult" in str(e):
                errors_mult[name].append((charge, multiplicity))
            else:
                errors_misc[str(e)[:30]][label].append((charge, multiplicity))
            continue

    elements.extend(list(set(qc_input['symbols'])))
    metal = [x for x in qc_input['symbols'] if x not in ['C', 'H', 'P', 'S', 'O', 'N', 'F', 'Cl', 'Br']][0]
    metals[metal].append(f"{label}")
    molecular_weights.append(qc_input['extras']["molecular_weight"])
    
elements = list(set(elements))
        

In [7]:
print(f"{len(molecules)} molecule/charge/multiplicity were imported.")

print("\nThe following errors DO remove molecules from the dataset:")
print(f"    There are {int(sum(sum([len(values) for _, values in x.items()]) for _, x in errors_misc.items()))} conformers with overlapping atoms.")

103032 molecule/charge/multiplicity were imported.

The following errors DO remove molecules from the dataset:
    There are 0 conformers with overlapping atoms.


## Assembled Dataset

In [8]:
dataset_name = "tmQM Charge Multiplicity Variant Optimization Dataset v0.0"
tagline = "BP86/def2-TZVP optimizations for tmQM-derived Pd, Zn, Fe, and Cu complexes across enumerated charge and multiplicity variants with charges of {-1,0,+1}."
description = """
This dataset was generated starting from the tmQM dataset (release 13Aug2024, https://github.com/uiocompcat/tmQM) containing 108541 unique molecules;
each molecule was evaluated using gfn2-xtb, and then a short MD simulation performed to provide additional configurations of the molecules. Further details
can be found in the hosting repository: https://zenodo.org/records/15059433. This dataset contains 23,134 unique transition metal complexes with one Pd, Zn, 
Fe, or Cu, and also only contain elements C, H, P, S, O, N, F, Cl, or Br with charges: {-1,0,+1}. Run with the BP86/def2-TZVP for loose optimizations generating
the properties: 'energy', 'gradient', 'dipole', 'quadrupole', 'wiberg_lowdin_indices', 'mayer_indices', 'lowdin_charges', 'lowdin_spins', 'dipole_polarizabilities',
'mulliken_charges'.
"""

dataset = client.add_dataset( # https://docs.qcarchive.molssi.org/user_guide/qcportal_reference.html
    "optimization", # collection type
    dataset_name, # Dataset name
    tagline=tagline,
    description=description,
    tags=["openff"],
    provenance={
        "qcportal": qcportal.__version__,
    },
    default_tag="openff",
    metadata={
        "submitter": "jaclark5",
        "creation_date": date.today(),
        'collection_type': 'OptimizationDataset',
        "long_description": description,
        'long_description_url': f'https://github.com/openforcefield/qca-dataset-submission/tree/master/submissions/2026-09-01-{dataset_name.replace(" ", "-")}',
        "short description": tagline,
        "dataset_name": dataset_name,
        "elements": elements,
    },
)

'metadata' parameter has been deprecated and will be removed in a future version. Use 'extras' instead


In [9]:
entries = []
for name, mol in molecules.items():
    dataset.add_entry(
        name=name, 
        initial_molecule=mol,
    )

Connection error for http://localhost:52191/api/v1/datasets/optimization/1/entries/bulkCreate: ('Connection aborted.', BadStatusLine('POST /api/v1/datasets/optimization/1/entries/bulkCreate HTTP/1.1\r\n')) - retrying in 0.51 seconds [1/5]
Connection error for http://localhost:52191/api/v1/datasets/optimization/1/entries/bulkCreate: ('Connection aborted.', BadStatusLine('POST /api/v1/datasets/optimization/1/entries/bulkCreate HTTP/1.1\r\n')) - retrying in 0.52 seconds [1/5]
Connection error for http://localhost:52191/api/v1/datasets/optimization/1/entries/bulkCreate: ('Connection aborted.', BadStatusLine('POST /api/v1/datasets/optimization/1/entries/bulkCreate HTTP/1.1\r\n')) - retrying in 0.49 seconds [1/5]
Connection error for http://localhost:52191/api/v1/datasets/optimization/1/entries/bulkFetch: ('Connection aborted.', BadStatusLine('POST /api/v1/datasets/optimization/1/entries/bulkFetch HTTP/1.1\r\n')) - retrying in 0.48 seconds [1/5]
Connection error for http://localhost:52191/ap

In [10]:
spec = QCSpecification(
    program='psi4',
    driver=SinglepointDriver.gradient,
    method='BP86',
    basis='def2-TZVP',
    keywords={
        'maxiter': 500, 
        'scf_properties': ['dipole', 'quadrupole', 'wiberg_lowdin_indices', 'mayer_indices', 'lowdin_charges', 'lowdin_spins', 'mulliken_charges'],
        'function_kwargs': {'properties': ['dipole_polarizabilities']},
        'reference': 'uks',
        'properties_origin': ['COM'],
    },
    protocols={'wavefunction': 'none'}
)
opt_spec = OptimizationSpecification(
    program="geometric",
    qc_specification=spec, 
    keywords={
        "tmax": 0.3,
        "check": 0,
        "qccnv": False,
        "reset": True,
        "trust": 0.1,
        "molcnv": False,
        "enforce": 0.0,
        "epsilon": 1e-05,
        "maxiter": 300,
        "coordsys": "dlc",
        "convergence_set": "GAU",
        "converge": ['energy', '1e-3', 'grms', '0.2', 'gmax', '1.0', 'drms', '15', 'dmax', '30'],
    },
)
dataset.add_specification(name="BP86/def2-TZVP", specification=opt_spec)

InsertMetadata(error_description=None, errors=[], inserted_idx=[0], existing_idx=[])

In [11]:
scaffold.to_json(dataset, compress=True)
#dataset.submit()

## Make Outputs

In [12]:
print("Elements:", ", ".join(dataset.extras["elements"]))
print("Charges:", Counter([float(c) for x in charges.values() for c in x]))
print("Multiplicities:", Counter([float(c) for x in multiplicities.values() for c in x]))
print("Metals:", {sym: len(set(x)) for sym, x in Counter(metals).items()})

print("Molecular Weight (min mean max):", int(np.min(molecular_weights)), int(np.mean(molecular_weights)), int(np.max(molecular_weights)))
            
print("Number of Molecules:", len(conformers))
print("Number of Conformers:", len(conformers))
n_conformers = np.array(list(conformers.values()))
print("Number of conformers (min mean max):", 1, 1, 1)

n_chg_mult = np.array([len(x) for x in charge_mult.values()])
print("Number of Charge/Multiplicity variations per molecule:", sum(n_chg_mult))
print("Number of Charge/Multiplicity variations per molecule (min mean max):", int(np.min(n_chg_mult)), int(np.mean(n_chg_mult)), int(np.max(n_chg_mult)))

Elements: Pd, Zn, N, C, H, Cu, S, Fe, O, Br, F, P, Cl
Charges: Counter({0.0: 40964, 1.0: 36530, -1.0: 25538})
Multiplicities: Counter({2.0: 30099, 1.0: 26513, 3.0: 17613, 4.0: 16030, 6.0: 7809, 5.0: 4968})
Metals: {'Cu': 3118, 'Pd': 9362, 'Zn': 6395, 'Fe': 4259}
Molecular Weight (min mean max): 95 589 2541
Number of Molecules: 23134
Number of Conformers: 23134
Number of conformers (min mean max): 1 1 1
Number of Charge/Multiplicity variations per molecule: 103032
Number of Charge/Multiplicity variations per molecule (min mean max): 1 4 9


In [13]:
for spec, obj in dataset.specifications.items():
    obj = obj.dict()
    obj = obj['specification']
    print(obj.keys())
    print("* Spec:", spec)
    print(f"    * program: {obj['program']}")
    print(f"    * keywords:")
    for k, v in obj["keywords"].items():
        print(f"       * {k}: {v}")
    print(f"    * qc_specification:")
    for k, field in obj['qc_specification'].items():
        print(f"       * {k}: {field}")
    print("* SCF properties:")
    for field in obj['qc_specification']['keywords']["scf_properties"]:
        print(f"       * {field}")

dict_keys(['program', 'qc_specification', 'keywords', 'protocols'])
* Spec: BP86/def2-TZVP
    * program: geometric
    * keywords:
       * tmax: 0.3
       * check: 0
       * qccnv: False
       * reset: True
       * trust: 0.1
       * molcnv: False
       * enforce: 0.0
       * epsilon: 1e-05
       * maxiter: 300
       * converge: ['energy', '1e-3', 'grms', '0.2', 'gmax', '1.0', 'drms', '15', 'dmax', '30']
       * coordsys: dlc
       * convergence_set: GAU
    * qc_specification:
       * program: psi4
       * driver: SinglepointDriver.deferred
       * method: bp86
       * basis: def2-tzvp
       * keywords: {'maxiter': 500, 'reference': 'uks', 'scf_properties': ['dipole', 'quadrupole', 'wiberg_lowdin_indices', 'mayer_indices', 'lowdin_charges', 'lowdin_spins', 'mulliken_charges'], 'function_kwargs': {'properties': ['dipole_polarizabilities']}, 'properties_origin': ['COM']}
       * protocols: {'wavefunction': <WavefunctionProtocolEnum.none: 'none'>, 'stdout': True, 'erro

/var/folders/dr/9nnhm1493kv7_s9r_wj_l_jm0000gn/T/ipykernel_17221/3110524096.py:2: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  obj = obj.dict()
